1. Distance ka formula

KNN generally Euclidean Distance use karta hai:

$$ d(P,Q)=\sqrt{(x_1-y_1)^2+(x_2-y_2)^2+\cdots+(x_n-y_n)^2} $$

Agar sirf 2 features hain:

Age    Fare
22     7
23     8
30     20

Aur hume kisi row ki Fare missing hai:

Age    Fare
24     ?

To Age = 24 ke basis par distance:

Row 1
$$ d = \sqrt{(24-22)^2} $$ $$ = \sqrt{4}=2 $$
Row 2
$$ d = \sqrt{(24-23)^2}=1 $$
Row 3
$$ d = \sqrt{(24-30)^2}=6 $$

To nearest rows:

Age 23 → distance 1  ← nearest
Age 22 → distance 2
Age 30 → distance 6
2. KNN Imputer missing value kaise nikalta hai?

Maan lo:

Age    Fare
22      7
23      8
24      ?
25      9

Aur:

KNNImputer(n_neighbors=2)

Matlab 2 nearest neighbors use karo.

Age = 24 ke nearest:

Age 23 → Fare 8
Age 25 → Fare 9

Ab missing Fare:

$$ \hat{x} = \frac{8+9}{2} $$ $$ \boxed{\hat{x}=8.5} $$

To:

Age    Fare
22      7
23      8
24     8.5   ← imputed
25      9
3. General KNN Imputation formula

Agar k nearest neighbors hain aur unki values:

$$ x_1,x_2,x_3,\ldots,x_k $$

to simple case me:

$$ \boxed{ \hat{x}=\frac{1}{k}\sum_{i=1}^{k}x_i } $$

Yaani nearest K rows ki values ka average.

4. Lekin sklearn ka KNN thoda important hai

KNNImputer distance ko use karke neighbors ko select karta hai, aur missing value estimate karte waqt distance-based weighting bhi use kar sakta hai.

Default:

KNNImputer(
    n_neighbors=5,
    weights='uniform'
)

weights='uniform' ka matlab:

Neighbor 1 → equal importance
Neighbor 2 → equal importance
Neighbor 3 → equal importance
...

Lekin:

weights='distance'

karoge to jo neighbor paas hai uska weight zyada hoga.

Formula:

$$ w_i=\frac{1}{d_i} $$

Aur weighted average:

$$ \boxed{ \hat{x}= \frac{\sum_{i=1}^{k}w_i x_i} {\sum_{i=1}^{k}w_i} } $$

jahan:

$$ w_i=\frac{1}{d_i} $$
Chhota example

2 neighbors:

Neighbor A:
Fare = 8
Distance = 1

Neighbor B:
Fare = 20
Distance = 4

Weights:

$$ w_A=\frac{1}{1}=1 $$ $$ w_B=\frac{1}{4}=0.25 $$

Ab:

$$ \hat{x} = \frac{(1)(8)+(0.25)(20)} {1+0.25} $$ $$ = \frac{8+5}{1.25} $$ $$ \boxed{10.4} $$

Dekho 8 wala neighbor paas tha, isliye uska influence zyada hua.

Sabse important difference
weights='uniform'

→ sab neighbors ko equal importance
→ simple average
weights='distance'

→ paas wala neighbor = zyada importance
→ door wala neighbor = kam importance

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [6]:
from pathlib import Path

print(list(Path("d:/Machine_learning").rglob("train.csv")))

[WindowsPath('d:/Machine_learning/data/train.csv'), WindowsPath('d:/Machine_learning/Feature_engineering/missingdata/train.csv')]


In [7]:
df = pd.read_csv('d:/Machine_learning/data/train.csv')[['Age','Pclass','Fare','Survived']]

In [8]:
df.head()

,Age,Pclass,Fare,Survived
0,22.0,3,7.2500,0
1,38.0,1,71.2833,1
2,26.0,3,7.9250,1
3,35.0,1,53.1000,1
4,35.0,3,8.0500,0


In [9]:
df.isnull().mean() * 100

Age         19.86532
Pclass       0.00000
Fare         0.00000
Survived     0.00000
dtype: float64

In [10]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [12]:
X_train.head()

,Age,Pclass,Fare
30,40.0,1,27.7208
10,4.0,3,16.7000
873,47.0,3,9.0000
182,9.0,3,31.3875
876,20.0,3,9.8458


In [ ]:
# knn = KNNImputer(n_neighbors=3,weights='distance')

# X_train_trf = knn.fit_transform(X_train)
# X_test_trf = knn.transform(X_test)

In [16]:
knn=KNNImputer(n_neighbors=3,weights="distance")
X_train_trf= knn.fit_transform(X_train)
X_test_trf= knn.transform(X_test)

In [17]:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred)

0.7039106145251397

In [18]:
# Comparision with Simple Imputer --> mean

si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [19]:
lr = LogisticRegression()

lr.fit(X_train_trf2,y_train)

y_pred2 = lr.predict(X_test_trf2)

accuracy_score(y_test,y_pred2)

0.6927374301675978